# Simple Linear Regression — Area vs Price

---

**Project:** ML with Scikit-Learn  
**Notebook:** 01 — Simple Linear Regression  
**Algorithm:** Simple Linear Regression (1 feature, 1 label)  
**Author:** Ather-ops  

---

## Objective

Build the simplest possible regression model — one input, one output. Given only the **Area** of a house in square feet, predict its **Price**. This is the foundation of all ML pipelines. Every concept here — EDA, outlier detection, scaling, training, evaluation, and prediction — applies directly to more complex models.

---

## The Model

Simple Linear Regression fits a straight line through the data:

$$\hat{Price} = w \times Area + b$$

Where:
- $w$ = slope — how much price increases per unit of area
- $b$ = intercept — base price when area is zero
- $\hat{Price}$ = predicted price

The model learns $w$ and $b$ by minimizing the **Mean Squared Error** across all training houses.

---

## Full Pipeline

| Step | Task |
|------|------|
| 1 | Import libraries |
| 2 | Create dataset |
| 3 | Exploratory Data Analysis |
| 4 | Descriptive statistics and missing value check |
| 5 | Outlier detection and treatment |
| 6 | Feature and target selection |
| 7 | Train-test split |
| 8 | Feature scaling |
| 9 | Model training |
| 10 | Predictions |
| 11 | Model evaluation |
| 12 | Visualization — regression line |
| 13 | Actual vs Predicted plot |
| 14 | Residual analysis |
| 15 | Predict price of a new house |
| 16 | New house prediction visualization |

---
## Step 1 — Import Libraries

| Library | Role |
|---------|------|
| `pandas` | Create and manipulate the DataFrame |
| `numpy` | Numerical operations, new house input |
| `matplotlib.pyplot` | All plots and visualizations |
| `train_test_split` | Split data into training and test sets |
| `LinearRegression` | The simple linear regression model |
| `StandardScaler` | Normalize Area feature before training |
| `mean_squared_error` | Average squared prediction error |
| `mean_absolute_error` | Average absolute prediction error |
| `r2_score` | How well the line fits the data (0 to 1) |

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

---
## Step 2 — Create Dataset

We create a simple, clean housing dataset manually. It contains 20 houses with two columns:

| Column | Type | Description |
|--------|------|-------------|
| `Area` | int | Size of the house in square feet |
| `Price` | int | Sale price of the house in thousands |

The dataset is deliberately small and realistic so every step of the pipeline is easy to follow and inspect. The general pattern is that larger area means higher price — with some natural noise added.

In [ ]:
# Create simple housing dataset — 20 houses, 1 feature, 1 label
data = {
    "Area": [600, 800, 900, 1000, 1100, 1200, 1400, 1500, 1600, 1800,
             2000, 2200, 2400, 2500, 2700, 3000, 3200, 3500, 3800, 4000],
    "Price": [120, 150, 160, 180, 195, 210, 240, 255, 270, 300,
              330, 360, 390, 405, 430, 470, 500, 540, 580, 610]
}

df = pd.DataFrame(data)

print("="*50)
print("Dataset — All 20 Houses:")
print("="*50)
print(df.to_string(index=True))
print("="*50)
print("Shape:", df.shape)

---
## Step 3 — Exploratory Data Analysis

Before building any model, we visualize the raw relationship between `Area` and `Price`.

**What to look for:**
- Is there a clear upward trend? If yes, linear regression is appropriate
- Are there any isolated dots far from the main cluster? Those are outliers
- Is the spread of points roughly consistent across all area values? If not, variance is non-constant (heteroscedasticity)

A scatter plot is always the first step. It takes 3 seconds and tells you more about the data than any statistic.

In [ ]:
# EDA — raw scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(df["Area"], df["Price"], color="steelblue", edgecolors="white",
            s=80, linewidth=0.8, label="Houses")
plt.xlabel("Area (sq ft)", fontsize=12)
plt.ylabel("Price (thousands)", fontsize=12)
plt.title("Area vs Price — Raw Data (Before Training)", fontsize=13)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*50)

---
## Step 4 — Descriptive Statistics and Missing Value Check

### `df.describe()` — Statistical Summary

| Statistic | What to check |
|-----------|---------------|
| `count` | Should be 20 for both columns — confirms no missing values |
| `mean` | Average area and average price — sanity check |
| `min` | Smallest area and price — should be positive |
| `max` | Largest values — compare with 75th percentile to spot outliers |
| `std` | High std relative to mean suggests wide spread or outliers |

### `df.isnull().sum()` — Missing Values

Any non-zero value here must be handled before training. Scikit-Learn raises a `ValueError` on `NaN` inputs.

In [ ]:
# Descriptive statistics
print("Basic Statistics:")
print("="*50)
print(df.describe())
print("="*50)
print("Missing Values per Column:")
print(df.isnull().sum())
print("="*50)
print("Data Types:")
print(df.dtypes)

---
## Step 5 — Outlier Detection using IQR

We apply the **Interquartile Range** method to detect outliers in the `Price` column.

**Formula:**

$$IQR = Q3 - Q1$$

$$\text{Lower Bound} = Q1 - 1.5 \times IQR$$

$$\text{Upper Bound} = Q3 + 1.5 \times IQR$$

Any price below `Lower Bound` or above `Upper Bound` is flagged as an outlier.

**Treatment — Clipping:**  
We clip rather than delete outliers. Clipping replaces extreme values with the fence boundary — keeping every row while removing the distortion.

In [ ]:
# IQR outlier detection on Price
Q1 = df["Price"].quantile(0.25)
Q3 = df["Price"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Outlier Detection — Price Column")
print("="*50)
print(f"Q1          : {Q1}")
print(f"Q3          : {Q3}")
print(f"IQR         : {IQR}")
print(f"Lower Bound : {lower_bound}")
print(f"Upper Bound : {upper_bound}")
print("="*50)

# Detect outliers
outliers = df[(df["Price"] < lower_bound) | (df["Price"] > upper_bound)]
print(f"Outliers found: {len(outliers)}")
if len(outliers) > 0:
    print(outliers)
print("="*50)

# Apply clipping
df["Price"] = np.clip(df["Price"], lower_bound, upper_bound)
print("Clipping applied. Price column cleaned.")
print("="*50)

---
## Step 5b — Outlier Visualization — Boxplot

A boxplot gives a visual representation of the IQR method. The box shows Q1 to Q3 (the middle 50% of prices). The whiskers extend to the fences. Any dot beyond the whiskers is an outlier.

If our dataset is clean, no dots should appear outside the whiskers after clipping.

In [ ]:
# Boxplot — Price distribution after clipping
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot
axes[0].boxplot(df["Price"], patch_artist=True,
                boxprops=dict(facecolor="steelblue", color="navy"),
                medianprops=dict(color="red", linewidth=2),
                whiskerprops=dict(color="navy"),
                capprops=dict(color="navy"))
axes[0].axhline(lower_bound, color="orange", linestyle="--",
                linewidth=1.2, label=f"Lower Bound: {lower_bound:.0f}")
axes[0].axhline(upper_bound, color="green", linestyle="--",
                linewidth=1.2, label=f"Upper Bound: {upper_bound:.0f}")
axes[0].set_ylabel("Price (thousands)")
axes[0].set_title("Price Boxplot — IQR Fences")
axes[0].legend(fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Histogram
axes[1].hist(df["Price"], bins=8, color="steelblue",
             edgecolor="white", linewidth=0.8)
axes[1].axvline(lower_bound, color="orange", linestyle="--",
                linewidth=1.5, label="Lower Bound")
axes[1].axvline(upper_bound, color="green", linestyle="--",
                linewidth=1.5, label="Upper Bound")
axes[1].set_xlabel("Price (thousands)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Price Distribution")
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("="*50)

---
## Step 6 — Feature and Target Selection

This is **Simple** Linear Regression — exactly one input feature and one output label.

| Variable | Column | Shape | Role |
|----------|--------|-------|------|
| `X` | Area | (20, 1) | Input feature matrix — must be 2D for Scikit-Learn |
| `Y` | Price | (20,) | Target vector — 1D |

**Why `df[["Area"]]` and not `df["Area"]`:**  
Scikit-Learn expects `X` to be a 2D array with shape `(n_samples, n_features)`. Using double brackets `[["Area"]]` gives shape `(20, 1)`. Single brackets `["Area"]` gives a 1D Series with shape `(20,)` which will raise a `ValueError`.

In [ ]:
# Feature and target selection
X = df[["Area"]]   # 2D — required by Scikit-Learn
Y = df["Price"]    # 1D target

print("Feature matrix X shape:", X.shape)
print("Target vector Y shape :", Y.shape)
print("="*50)
print("X (first 5 rows):")
print(X.head())
print("="*50)
print("Y (first 5 values):")
print(Y.head().to_string())
print("="*50)

---
## Step 7 — Train-Test Split

We split the 20 houses into training and test sets.

| Parameter | Value | Effect |
|-----------|-------|--------|
| `test_size=0.2` | 20% | 4 houses held back for testing |
| `random_state=42` | 42 | Reproducible — same split every run |

With 20 houses: 16 go to training, 4 go to testing.

The model trains on 16 houses. It is then evaluated on the remaining 4 — houses it has never seen during training. This gives an honest measure of generalization.

In [ ]:
# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("Train-Test Split")
print("="*50)
print(f"Total houses  : {len(X)}")
print(f"Training set  : {len(X_train)} houses")
print(f"Test set      : {len(X_test)} houses")
print("="*50)
print("X_train shape :", X_train.shape)
print("X_test shape  :", X_test.shape)
print("Y_train shape :", Y_train.shape)
print("Y_test shape  :", Y_test.shape)
print("="*50)

---
## Step 8 — Feature Scaling

Area ranges from 600 to 4000. StandardScaler transforms it to have mean = 0 and std = 1.

$$z = \frac{x - \mu}{\sigma}$$

For Simple Linear Regression on a single feature, scaling has less mathematical impact than in multi-feature models. However it is included here because:
- It is essential practice for any ML pipeline
- It ensures the intercept term is properly computed
- When this notebook scales to multiple features, the pattern is already correct

**Rule:** Fit on training data only. Apply the same transformation to test data.

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("StandardScaler Applied")
print("="*50)
print(f"Learned mean  : {scaler.mean_[0]:.2f}")
print(f"Learned std   : {scaler.scale_[0]:.2f}")
print("="*50)
print("X_train_scaled (first 5):")
print(X_train_scaled[:5].round(4))
print("="*50)

---
## Step 9 — Model Training

We initialize `LinearRegression` and call `.fit()` on the scaled training data.

**What happens during `.fit()`:**

Scikit-Learn solves the Normal Equation:

$$\theta = (X^T X)^{-1} X^T Y$$

For one feature this reduces to finding the single slope $w$ and intercept $b$ that produce the line of best fit through the 16 training houses.

After training we print the learned slope and intercept. The slope tells us: for every 1-unit increase in scaled Area, Price increases by `coef_` thousand units.

In [ ]:
# Model training
model = LinearRegression()
model.fit(X_train_scaled, Y_train)

print("Model Trained")
print("="*50)
print(f"Slope (w)     : {model.coef_[0]:.4f}")
print(f"Intercept (b) : {model.intercept_:.4f}")
print("="*50)
print("Equation: Price = {:.4f} x Area_scaled + {:.4f}".format(
    model.coef_[0], model.intercept_
))

---
## Step 10 — Predictions

We run the trained model on the scaled test set to generate predicted prices for the 4 test houses.

We then print a comparison table — actual price vs predicted price for each test house. This gives an immediate, human-readable sense of how close the model is before looking at aggregate metrics.

In [ ]:
# Predictions on test set
y_pred = model.predict(X_test_scaled)

# Comparison table
results = pd.DataFrame({
    "Area"          : X_test["Area"].values,
    "Actual Price"  : Y_test.values,
    "Predicted Price": y_pred.round(2),
    "Error"         : (Y_test.values - y_pred).round(2)
})

print("Prediction Results — Test Set")
print("="*60)
print(results.to_string(index=False))
print("="*60)

---
## Step 11 — Model Evaluation

Three standard regression metrics measure how well the model performs on the test set:

| Metric | Formula | Unit | Interpretation |
|--------|---------|------|----------------|
| MSE | $\frac{1}{n}\sum(y - \hat{y})^2$ | Price squared | Lower is better — penalizes large errors heavily |
| RMSE | $\sqrt{MSE}$ | Price (thousands) | Same unit as Price — most interpretable |
| MAE | $\frac{1}{n}\sum|y - \hat{y}|$ | Price (thousands) | Average absolute error per house |
| R2 | $1 - \frac{SS_{res}}{SS_{tot}}$ | Unitless 0 to 1 | Proportion of price variance explained by Area |

**R2 for Simple Linear Regression:**  
An R2 of 0.95 means that 95% of all price variation across houses is explained purely by their area. The remaining 5% comes from factors not in our model — location, condition, number of rooms, etc.

In [ ]:
# Model evaluation
MSE      = mean_squared_error(Y_test, y_pred)
RMSE     = np.sqrt(MSE)
MAE      = mean_absolute_error(Y_test, y_pred)
R2       = r2_score(Y_test, y_pred)

print("Model Evaluation — Test Set")
print("="*50)
print(f"MSE      : {MSE:.4f}")
print(f"RMSE     : {RMSE:.4f}  (in price units)")
print(f"MAE      : {MAE:.4f}  (in price units)")
print(f"R2 Score : {R2:.4f}")
print("="*50)
print(f"The model explains {R2*100:.1f}% of price variation using Area alone.")

---
## Step 12 — Regression Line Visualization

This is the core plot of Simple Linear Regression. We plot:
- All training data points as blue dots
- All test data points as orange dots
- The learned regression line across the full area range as a red line

**Reading this plot:**
- The red line is the model's learned equation $\hat{Price} = w \times Area + b$
- Points close to the line = small error
- Points far from the line = large error (residuals)
- A good model has the line passing through the center of the point cloud

In [ ]:
# Regression line visualization
# Generate area range for drawing the line
area_range      = np.linspace(df["Area"].min(), df["Area"].max(), 200).reshape(-1, 1)
area_range_scaled = scaler.transform(area_range)
price_line      = model.predict(area_range_scaled)

plt.figure(figsize=(9, 5))

# Training points
plt.scatter(X_train["Area"], Y_train,
            color="steelblue", edgecolors="white",
            s=80, linewidth=0.8, label="Training houses", zorder=3)

# Test points
plt.scatter(X_test["Area"], Y_test,
            color="darkorange", edgecolors="white",
            s=100, linewidth=0.8, marker="D", label="Test houses", zorder=4)

# Regression line
plt.plot(area_range, price_line,
         color="red", linewidth=2, label="Regression line", zorder=2)

plt.xlabel("Area (sq ft)", fontsize=12)
plt.ylabel("Price (thousands)", fontsize=12)
plt.title("Simple Linear Regression — Area vs Price", fontsize=13)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*50)

---
## Step 13 — Actual vs Predicted Plot

We plot actual test prices on the X-axis against predicted prices on the Y-axis.

**How to read it:**
- A perfect model places all points on the diagonal dashed line where Actual = Predicted
- Points above the line: model underestimated price
- Points below the line: model overestimated price
- The tighter the cluster around the diagonal, the higher the R2 score

In [ ]:
# Actual vs Predicted scatter
plt.figure(figsize=(6, 5))

plt.scatter(Y_test, y_pred,
            color="steelblue", edgecolors="white",
            s=120, linewidth=0.8, zorder=3, label="Test houses")

# Perfect prediction diagonal
min_val = min(Y_test.min(), y_pred.min()) - 10
max_val = max(Y_test.max(), y_pred.max()) + 10
plt.plot([min_val, max_val], [min_val, max_val],
         color="red", linestyle="--", linewidth=1.5,
         label="Perfect prediction", zorder=2)

plt.xlabel("Actual Price (thousands)", fontsize=12)
plt.ylabel("Predicted Price (thousands)", fontsize=12)
plt.title("Actual vs Predicted Price", fontsize=13)
plt.legend(fontsize=10)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*50)

---
## Step 14 — Residual Analysis

**Residuals** = Actual Price minus Predicted Price

Residual analysis is a standard diagnostic for linear regression. It checks one of the core assumptions of the model — that errors are randomly distributed with no systematic pattern.

| Plot | What it shows |
|------|---------------|
| Residuals vs Area | Are errors random across all area values? |
| Residual histogram | Are errors normally distributed around zero? |

**What to look for:**
- Residuals should scatter randomly around the zero line with no pattern
- The histogram should resemble a bell curve centered at zero
- A curve or funnel shape in residuals vs area = the relationship is non-linear or has non-constant variance

In [ ]:
# Residual analysis
residuals = Y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs Area
axes[0].scatter(X_test["Area"], residuals,
                color="steelblue", edgecolors="white",
                s=100, linewidth=0.8)
axes[0].axhline(0, color="red", linestyle="--", linewidth=1.5)
axes[0].set_xlabel("Area (sq ft)", fontsize=11)
axes[0].set_ylabel("Residual (Actual - Predicted)", fontsize=11)
axes[0].set_title("Residuals vs Area", fontsize=12)
axes[0].grid(True, linestyle="--", alpha=0.5)

# Residual histogram
axes[1].hist(residuals, bins=6, color="steelblue",
             edgecolor="white", linewidth=0.8)
axes[1].axvline(0, color="red", linestyle="--",
                linewidth=1.5, label="Zero error")
axes[1].set_xlabel("Residual", fontsize=11)
axes[1].set_ylabel("Frequency", fontsize=11)
axes[1].set_title("Residual Distribution", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()
print("="*50)
print(f"Mean residual  : {residuals.mean():.4f}  (should be close to 0)")
print(f"Std of residuals: {residuals.std():.4f}")
print("="*50)

---
## Step 15 — Predict Price of a New House

The trained model can now predict the price of any house given only its area — including a house it has never seen.

**New house:**

| Feature | Value |
|---------|-------|
| Area | 2800 sq ft |

**Pipeline for new input:**
1. Create the input as a NumPy array with shape `(1, 1)`
2. Scale using the **same scaler** — `transform()` only, never `fit_transform()`
3. Pass through `model.predict()` to get the predicted price

In [ ]:
# Predict price for a new house
new_house_area    = np.array([[2800]])
new_house_scaled  = scaler.transform(new_house_area)
predicted_price   = model.predict(new_house_scaled)

print("New House Prediction")
print("="*50)
print(f"Input Area      : {new_house_area[0][0]} sq ft")
print(f"Predicted Price : {predicted_price[0]:.2f} thousand")
print("="*50)

---
## Step 16 — New House Prediction Visualization

We place the new house prediction on the regression line plot so it is visible in context. The large red star marks where the new house sits — its area on the X-axis and its predicted price on the Y-axis.

This answers the most natural question visually: does the predicted price look reasonable given where this house falls relative to all other houses on the regression line?

In [ ]:
# New house prediction — visualization in context
plt.figure(figsize=(9, 5))

# All data points
plt.scatter(X_train["Area"], Y_train,
            color="steelblue", edgecolors="white",
            s=70, linewidth=0.8, label="Training houses", zorder=3)
plt.scatter(X_test["Area"], Y_test,
            color="darkorange", edgecolors="white",
            s=90, linewidth=0.8, marker="D", label="Test houses", zorder=4)

# Regression line
plt.plot(area_range, price_line,
         color="red", linewidth=2, label="Regression line", zorder=2)

# New house — star marker
plt.scatter(2800, predicted_price[0],
            color="red", marker="*", s=400, zorder=5,
            label=f"New house (2800 sqft) = {predicted_price[0]:.1f}K")

# Drop lines to axes
plt.vlines(2800, 0, predicted_price[0],
           colors="gray", linestyles="dotted", linewidth=1.2)
plt.hlines(predicted_price[0], df["Area"].min(), 2800,
           colors="gray", linestyles="dotted", linewidth=1.2)

plt.xlabel("Area (sq ft)", fontsize=12)
plt.ylabel("Price (thousands)", fontsize=12)
plt.title(
    f"New House Prediction — Area: 2800 sqft  |  Predicted Price: {predicted_price[0]:.2f}K",
    fontsize=12
)
plt.legend(fontsize=9)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
print("="*50)

---
## Pipeline Summary

| Step | Action | Output |
|------|--------|--------|
| 1 | Import libraries | All tools ready |
| 2 | Create dataset | 20 houses, Area and Price columns |
| 3 | EDA scatter plot | Raw Area vs Price relationship visualized |
| 4 | Descriptive statistics and missing value check | Data quality confirmed |
| 5 | IQR outlier detection and clipping | Price column cleaned |
| 5b | Boxplot and histogram | Outlier treatment confirmed visually |
| 6 | Feature and target selection | X shape (20,1), Y shape (20,) |
| 7 | Train-test split 80/20 | 16 training houses, 4 test houses |
| 8 | StandardScaler — fit on train, transform both | X_train_scaled, X_test_scaled |
| 9 | LinearRegression trained | Slope and intercept learned |
| 10 | Predictions on test set | y_pred — 4 predicted prices |
| 11 | Evaluation — MSE, RMSE, MAE, R2 | Model performance quantified |
| 12 | Regression line plot | Visual fit of learned line |
| 13 | Actual vs Predicted plot | Per-house prediction accuracy |
| 14 | Residual analysis | Error distribution validated |
| 15 | New house prediction | Price predicted for 2800 sqft house |
| 16 | New house on regression plot | Prediction placed in context |

---

**Next notebook:** `02_multiple_linear_regression.ipynb` — Adding more features (Bedrooms, Age) to the same pipeline